# Ingredient parse — error analysis

Browse flagged rows from `scratch/parse_experiments/{batch_id}/`.

**Kernel cwd:** Capstone project root (`Capstone/`) or this notebook's folder (`scratch/`).

**Error categories** (pipe-separated when multiple apply):
- `unparsed_with_leading_qty` — library failed but raw line looks parseable
- `llm_resolved` — final answer came from LLM
- `low_llm_certainty` — LLM certainty &lt; 0.7
- `llm_error` / `llm_invalid` — LLM failures
- `rules_rescued` — Kadin rules fixed a library miss (`lib_rules_llm` only)
- `disagreement_vs_other` — final fields differ from the other pipeline

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
def resolve_parse_experiments_dir() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd / "scratch" / "parse_experiments",
        cwd / "parse_experiments",
    ]
    for path in candidates:
        if path.is_dir():
            return path
    raise FileNotFoundError(
        "Could not find parse_experiments/. Set kernel cwd to Capstone/ or scratch/."
    )


PARSE_EXPERIMENTS = resolve_parse_experiments_dir()
BATCHES = sorted(p.name for p in PARSE_EXPERIMENTS.iterdir() if p.is_dir())
print(f"Artifacts root: {PARSE_EXPERIMENTS}")
print(f"Available batches ({len(BATCHES)}):")
for batch_id in BATCHES:
    print(f"  - {batch_id}")

In [ ]:
# Change this to any batch_id from the list above (default: latest)
BATCH_ID = BATCHES[-1] if BATCHES else None
BATCH_DIR = PARSE_EXPERIMENTS / BATCH_ID

with (BATCH_DIR / "eval_manifest.json").open() as f:
    MANIFEST = json.load(f)

with (BATCH_DIR / "comparison" / "comparison_summary.json").open() as f:
    COMPARISON = json.load(f)

with (BATCH_DIR / "lib_llm" / "parse_summary.json").open() as f:
    SUMMARY_V1 = json.load(f)

with (BATCH_DIR / "lib_rules_llm" / "parse_summary.json").open() as f:
    SUMMARY_V2 = json.load(f)

print(f"Batch: {BATCH_ID}")
print(
    f"  recipes={MANIFEST['n_recipes']}  lines={MANIFEST['n_ingredient_lines']}  "
    f"unique={MANIFEST['n_unique_ingredients']}  model={MANIFEST['model']}"
)
print(
    f"  lib_llm coverage_ok={SUMMARY_V1['coverage_ok_rate']:.2%}  "
    f"llm_calls={SUMMARY_V1['n_llm_calls']}  cost=${SUMMARY_V1['cost_total_usd']:.4f}"
)
print(
    f"  lib_rules_llm coverage_ok={SUMMARY_V2['coverage_ok_rate']:.2%}  "
    f"llm_calls={SUMMARY_V2['n_llm_calls']}  cost=${SUMMARY_V2['cost_total_usd']:.4f}"
)
print(f"  disagreement_rate={COMPARISON['disagreement_rate']:.2%}  rules_rescued={COMPARISON['rules_rescued_count']}")

In [ ]:
def load_batch_tables(batch_dir: Path) -> dict[str, pd.DataFrame]:
    tables: dict[str, pd.DataFrame] = {}
    paths = {
        "errors_v1": batch_dir / "lib_llm" / "error_analysis.parquet",
        "errors_v2": batch_dir / "lib_rules_llm" / "error_analysis.parquet",
        "results_v1": batch_dir / "lib_llm" / "parse_results.parquet",
        "results_v2": batch_dir / "lib_rules_llm" / "parse_results.parquet",
        "llm_calls_v1": batch_dir / "lib_llm" / "llm_calls.parquet",
        "llm_calls_v2": batch_dir / "lib_rules_llm" / "llm_calls.parquet",
        "side_by_side": batch_dir / "comparison" / "side_by_side.parquet",
        "disagreements": batch_dir / "comparison" / "disagreements.parquet",
        "rules_rescued": batch_dir / "comparison" / "rules_rescued.parquet",
        "llm_only_fixes": batch_dir / "comparison" / "llm_only_fixes.parquet",
    }
    for name, path in paths.items():
        tables[name] = pd.read_parquet(path) if path.exists() else pd.DataFrame()
    return tables


TABLES = load_batch_tables(BATCH_DIR)
for key, df in TABLES.items():
    print(f"{key:16s} {len(df):>6,} rows")

## Category breakdown

In [ ]:
def split_categories(series: pd.Series) -> pd.Series:
    """Explode pipe-separated error_category values for counting."""
    exploded = (
        series.dropna()
        .astype(str)
        .str.split("|")
        .explode()
    )
    return exploded.value_counts()


print("lib_llm error categories:")
display(split_categories(TABLES["errors_v1"]["error_category"]))

print("lib_rules_llm error categories:")
display(split_categories(TABLES["errors_v2"]["error_category"]))

## Filter flagged rows

Set `PIPELINE` and `CATEGORY`. Use `None` for category to show all flagged rows. Partial matches work (e.g. `"llm_resolved"` matches `low_llm_certainty|llm_resolved`).

In [ ]:
PIPELINE = "lib_llm"  # lib_llm | lib_rules_llm
CATEGORY = "unparsed_with_leading_qty"  # or None for all flagged rows
SEARCH = ""  # optional substring on ingredient_raw
RECIPE_ID = None  # optional int filter
LIMIT = 25

DISPLAY_COLS = [
    "ingredient_raw",
    "recipe_id",
    "ingredient_idx",
    "error_category",
    "library_parse_status",
    "library_quantity",
    "library_unit",
    "library_name",
    "final_method",
    "final_parse_status",
    "final_quantity",
    "final_unit",
    "final_name",
    "llm_called",
    "llm_certainty",
    "llm_measurable",
    "llm_rationale",
]

RULES_COLS = [
    "rules_parse_status",
    "rules_quantity",
    "rules_unit",
    "rules_confidence",
]


def filter_errors(
    df: pd.DataFrame,
    *,
    category: str | None = None,
    search: str = "",
    recipe_id: int | None = None,
) -> pd.DataFrame:
    out = df.copy()
    if category:
        out = out[out["error_category"].astype(str).str.contains(category, regex=False, na=False)]
    if search:
        out = out[out["ingredient_raw"].astype(str).str.contains(search, case=False, na=False)]
    if recipe_id is not None:
        out = out[out["recipe_id"] == recipe_id]
    return out


errors = TABLES["errors_v1"] if PIPELINE == "lib_llm" else TABLES["errors_v2"]
cols = DISPLAY_COLS + (RULES_COLS if PIPELINE == "lib_rules_llm" else [])
cols = [c for c in cols if c in errors.columns]

filtered = filter_errors(errors, category=CATEGORY, search=SEARCH, recipe_id=RECIPE_ID)
print(f"{PIPELINE}: {len(filtered):,} rows match filters")
display(filtered[cols].head(LIMIT))

## Readable row inspector

Pick a row index from the filtered table above (0-based within `filtered`).

In [ ]:
ROW_IDX = 0


def show_error_row(row: pd.Series) -> None:
    print(f"raw: {row['ingredient_raw']}")
    print(f"recipe_id={row['recipe_id']}  idx={row['ingredient_idx']}  category={row['error_category']}")
    print("-" * 72)
    print(
        f"library: status={row.get('library_parse_status')}  "
        f"qty={row.get('library_quantity')}  unit={row.get('library_unit')}  name={row.get('library_name')}"
    )
    if "rules_parse_status" in row.index and pd.notna(row.get("rules_parse_status")):
        print(
            f"rules:   status={row.get('rules_parse_status')}  "
            f"qty={row.get('rules_quantity')}  unit={row.get('rules_unit')}  "
            f"conf={row.get('rules_confidence')}"
        )
    if row.get("llm_called"):
        print(
            f"llm:     qty={row.get('llm_quantity')}  unit={row.get('llm_unit')}  "
            f"name={row.get('llm_name')}  measurable={row.get('llm_measurable')}  "
            f"certainty={row.get('llm_certainty')}"
        )
        if pd.notna(row.get("llm_rationale")):
            print(f"rationale: {row.get('llm_rationale')}")
        if pd.notna(row.get("llm_error")):
            print(f"llm_error: {row.get('llm_error')}")
    print(
        f"final ({row.get('final_method')}): status={row.get('final_parse_status')}  "
        f"qty={row.get('final_quantity')}  unit={row.get('final_unit')}  name={row.get('final_name')}"
    )


if filtered.empty:
    print("No rows in filtered set.")
else:
    show_error_row(filtered.iloc[ROW_IDX])

## Pipeline comparison on the same ingredient

In [ ]:
COMPARE_COLS = [
    "ingredient_raw",
    "final_method_v1",
    "final_parse_status_v1",
    "final_quantity_v1",
    "final_unit_v1",
    "final_name_v1",
    "final_method_v2",
    "final_parse_status_v2",
    "final_quantity_v2",
    "final_unit_v2",
    "final_name_v2",
]

side = TABLES["side_by_side"]
if filtered.empty:
    print("No filtered rows to compare.")
else:
    row = filtered.iloc[ROW_IDX]
    key = (int(row["recipe_id"]), int(row["ingredient_idx"]))
    match = side[
        (side["recipe_id"] == key[0]) & (side["ingredient_idx"] == key[1])
    ]
    display(match[[c for c in COMPARE_COLS if c in match.columns]])

## Cross-pipeline slices

In [ ]:
DISAGREE_COLS = [
    "ingredient_raw",
    "final_quantity_v1",
    "final_unit_v1",
    "final_name_v1",
    "final_parse_status_v1",
    "final_quantity_v2",
    "final_unit_v2",
    "final_name_v2",
    "final_parse_status_v2",
]

print("Disagreements (v1 vs v2 final fields):")
display(TABLES["disagreements"][[c for c in DISAGREE_COLS if c in TABLES["disagreements"].columns]].head(20))

print("Rules rescued (lib_rules_llm fixed library miss):")
rescued_cols = [
    "ingredient_raw",
    "library_parse_status",
    "library_quantity",
    "library_unit",
    "final_method",
    "final_parse_status",
    "final_quantity",
    "final_unit",
    "final_name",
]
display(TABLES["rules_rescued"][[c for c in rescued_cols if c in TABLES["rules_rescued"].columns]])

print("LLM-only fixes with measurable ok outcome:")
display(TABLES["llm_only_fixes"][[c for c in DISAGREE_COLS if c in TABLES["llm_only_fixes"].columns]].head(20))

## LLM call log (deduped by normalized ingredient)

In [ ]:
LLM_PIPELINE = "lib_llm"  # lib_llm | lib_rules_llm
llm_calls = TABLES["llm_calls_v1"] if LLM_PIPELINE == "lib_llm" else TABLES["llm_calls_v2"]

llm_cols = [
    "ingredient_raw",
    "ingredient_norm",
    "llm_certainty",
    "llm_measurable",
    "llm_rationale",
    "llm_error",
    "fields",
]
llm_cols = [c for c in llm_cols if c in llm_calls.columns]

llm_filtered = llm_calls
if SEARCH:
    llm_filtered = llm_filtered[
        llm_filtered["ingredient_raw"].astype(str).str.contains(SEARCH, case=False, na=False)
    ]

print(f"{LLM_PIPELINE}: {len(llm_filtered):,} LLM calls")
display(llm_filtered[llm_cols].sort_values("llm_certainty").head(25))

## Still-unparsed lines (full results, not just flagged)

In [ ]:
results = TABLES["results_v1"] if PIPELINE == "lib_llm" else TABLES["results_v2"]

unparsed = results[
    ~results["final_parse_status"].isin(["ok", "unmeasurable"])
].copy()

unparsed_cols = [
    "ingredient_raw",
    "recipe_id",
    "library_parse_status",
    "final_method",
    "final_parse_status",
    "final_quantity",
    "final_unit",
    "final_name",
]
unparsed_cols = [c for c in unparsed_cols if c in unparsed.columns]

print(f"{PIPELINE}: {len(unparsed):,} lines not ok/unmeasurable")
display(unparsed[unparsed_cols].head(30))